In [1]:
# Here, we are saving the output value of the root finding function
# for different values of (N_tilde and t_Q) in a csv file

import numpy as np
import pandas as pd
from scipy.optimize import root_scalar as rs
import astropy
from astropy.cosmology import FlatLambdaCDM

Planck = astropy.cosmology.realizations.Planck18

# T_CMB = Planck.Tcmb0  # Temperature of the CMB
# H_0 = Planck.H0  # Current Hubble constant
# Omega_m = Planck.Om0  # Matter density parameter
# Omega_lambda = Planck.Ode0  # Dark energy density parameter
c = astropy.constants.c.to('km/s')  # Speed of light in km/s
# sigma_T = astropy.constants.sigma_T.to('km2')  # Thomson scattering cross-section in km^2
Omega_b = Planck.Ob0  # Baryon density parameter
# h = Planck.h  # Dimensionless Hubble parameter
rho_crit_0 = Planck.critical_density0.to('kg/m3')  # Critical density of the universe at z = 0 in kg/m^3
z = 8

In [2]:
# Setting up values and limits

N_tilde = np.linspace(0.001, 1000, int(1001))  # N_tilde: 0 to 1000
t_Q = np.linspace(0.001, 1000, int(1001))  # t_Q: 0 to 1000

# Setting up constants
n_H = rho_crit_0/(astropy.constants.m_p) * (1 - 0.247) * (Omega_b) * (1 + z)**3
n_H = n_H.to('km-3')

In [3]:
# Defining the root finding function

def radius_calc(theta, n_H): 
    N_tilde, t_tilde = theta
    return (((3)/(4*np.pi*n_H))**(1/3)) * (N_tilde * t_tilde)**(1/3) * (1e58)**(1/3) * (1e7 * 3.156e7)**(1/3)  

def root_func(t, theta, n_H, c = c):
    N_tilde, t_tilde = theta
    N_dot = N_tilde * 1e58  # in s^-1
    t_Q = t_tilde * 1e7 * 3.156e7  # in seconds

    return (t * 1e7 * 3.156e7) + (radius_calc([N_tilde, t], n_H).value)/(c.value) - t_Q

In [4]:
# Finding the root for each combination of N_tilde and t_Q and saving the results in a csv file
# adding a progress bar using tqdm
from tqdm import tqdm
results = []
for N in tqdm(N_tilde):
    for t in tqdm(t_Q):
        root = rs(root_func, args=([N, t], n_H), bracket=[1e-40, 1e40]).root
        results.append((N, t, root))

results

100%|██████████| 1001/1001 [1:07:27<00:00,  4.04s/it]


[(np.float64(0.001), np.float64(0.001), 1.5528008710863218e-07),
 (np.float64(0.001), np.float64(1.000999), 0.826431676926444),
 (np.float64(0.001), np.float64(2.000998), 1.7757365816315085),
 (np.float64(0.001), np.float64(3.000997), 2.7406736785567514),
 (np.float64(0.001), np.float64(4.000996), 3.712946669007109),
 (np.float64(0.001), np.float64(5.0009950000000005), 4.689627404104479),
 (np.float64(0.001), np.float64(6.000994), 5.669300376211616),
 (np.float64(0.001), np.float64(7.000993), 6.651160798611673),
 (np.float64(0.001), np.float64(8.000992), 7.634702330707398),
 (np.float64(0.001), np.float64(9.000990999999999), 8.619583356836271),
 (np.float64(0.001), np.float64(10.00099), 9.605561273542001),
 (np.float64(0.001), np.float64(11.000988999999999), 10.59245688526733),
 (np.float64(0.001), np.float64(12.000988), 11.580133646232603),
 (np.float64(0.001), np.float64(13.000986999999999), 12.568484844996012),
 (np.float64(0.001), np.float64(14.000986), 13.557425319844826),
 (np.fl

In [6]:
# Converting the results to a pandas dataframe
df = pd.DataFrame(results, columns=['N_tilde', 't_Q', 'root'])
# Saving the dataframe to a csv file
df.to_csv('root_finding_results.csv', index=False)
 

print(df)

          N_tilde          t_Q          root
0           0.001     0.001000  1.552801e-07
1           0.001     1.000999  8.264317e-01
2           0.001     2.000998  1.775737e+00
3           0.001     3.000997  2.740674e+00
4           0.001     4.000996  3.712947e+00
...           ...          ...           ...
1001996  1000.000   996.000004  8.217621e+02
1001997  1000.000   997.000003  8.226961e+02
1001998  1000.000   998.000002  8.236302e+02
1001999  1000.000   999.000001  8.245643e+02
1002000  1000.000  1000.000000  8.254984e+02

[1002001 rows x 3 columns]


In [8]:
rs(root_func, args=([1000, 996.5], n_H), bracket=[1e-40, 1e40]).root

822.229084717795